In [0]:
%pip install pymupdf python-docx

In [0]:
import io
import fitz  # PyMuPDF
import docx
from pyspark.sql.functions import udf, col, lit
from pyspark.sql.types import StructType, StructField, StringType, LongType

# Define Schema for the extraction output
extracted_schema = StructType([
    StructField("title", StringType(), True),
    StructField("content", StringType(), True),
    StructField("status", StringType(), True)
])

def extract_document_data(file_bytes, file_path):
    """
    Parses raw binary files based on extension to extract actual document titles and full text.
    """
    title = "Unknown Title"
    content = ""
    status = "Success"
    
    try:
        if file_bytes is None or len(file_bytes) == 0:
            return (title, content, "Empty File")
            
        # Handle PDF Documents
        if file_path.lower().endswith('.pdf'):
            with fitz.open(stream=file_bytes, filetype="pdf") as doc:
                # Fallback hierarchy for title extraction: Metadata -> Largest Text Font -> First Line
                meta_title = doc.metadata.get("title")
                if meta_title and meta_title.strip():
                    title = meta_title.strip()
                else:
                    # Traversal for largest font block on page 1 (structural title discovery)
                    largest_font = 0
                    fallback_title = ""
                    if len(doc) > 0:
                        page = doc[0]
                        blocks = page.get_text("dict")["blocks"]
                        for b in blocks:
                            if "lines" in b:
                                for l in b["lines"]:
                                    for s in l["spans"]:
                                        if s["size"] > largest_font and s["text"].strip():
                                            largest_font = s["size"]
                                            fallback_title = s["text"]
                        title = fallback_title.strip() if fallback_title else page.get_text().split('\n')[0]
                
                # Full text gathering
                text_layers = [page.get_text() for page in doc]
                content = " ".join(text_layers)

        # Handle Word Documents (.docx)
        elif file_path.lower().endswith('.docx'):
            doc_stream = io.BytesIO(file_bytes)
            doc = docx.Document(doc_stream)
            
            # Inspect paragraphs for semantic 'Title' styles
            for para in doc.paragraphs:
                if para.style.name == 'Title' and para.text.strip():
                    title = para.text.strip()
                    break
            
            if title == "Unknown Title" and len(doc.paragraphs) > 0:
                title = doc.paragraphs[0].text.strip()
                
            content = " ".join([para.text for para in doc.paragraphs])
            
        else:
            status = "Unsupported Extension"
            
    except Exception as e:
        status = f"Error: {str(e)}"
        
    return (title if title else "Untitled", content, status)

# Registering User Defined Function (UDF) for Distributed Execution
extract_docs_udf = udf(extract_document_data, extracted_schema)

In [0]:
import time

# Update these targets to map directly to your mounted cloud storage layer
SOURCE_DOCUMENTS_DIR = "/Volumes/test/default/raw_docs_volume/"
OUTPUT_HIGHLIGHTED_DIR = "/Volumes/test/default/raw_docs_volume/highlighted_output/"

print("[Status] Initializing Raw Document Ingestion from Cloud Storage Data Lake...")

# Step 1: Read raw files globally as binary frames
raw_binary_df = spark.read.format("binaryFile") \
    .option("pathGlobFilter", "*.*") \
    .option("recursiveFileLookup", "true") \
    .load(SOURCE_DOCUMENTS_DIR)

# Step 2: Execute parallel structural parsing
parsed_df = raw_binary_df.withColumn("extracted_data", extract_docs_udf(col("content"), col("path")))

# Step 3: Flatten Schema for consumer pipelines
documents_base_df = parsed_df.select(
    col("path"),
    col("modificationTime"),
    col("length").alias("file_size_bytes"),
    col("extracted_data.title").alias("document_title"),
    col("extracted_data.content").alias("document_text"),
    col("extracted_data.status").alias("processing_status")
).filter(col("processing_status") == "Success")

# Trigger execution plan evaluation via action
total_docs = documents_base_df.count()
print(f"[Success] Pipeline materialized. Total Successfully Parsed Documents: {total_docs}")

[Status] Initializing Raw Document Ingestion from Cloud Storage Data Lake...
[Success] Pipeline materialized. Total Successfully Parsed Documents: 20


In [0]:
def execute_document_sorting(df):
    """
    Sorts datasets across nodes using document title metadata strings.
    Measures complete end-to-end processing execution time.
    """
    start_time = time.time()
    
    # Distributed Sorting Transformation Order
    sorted_execution_df = df.orderBy(col("document_title").asc_nulls_last())
    
    # Force action execution to ensure accurate compute duration measurement
    execution_count = sorted_execution_df.count()
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    
    return sorted_execution_df, elapsed_time

# Run Sorting
sorted_df, sort_duration = execute_document_sorting(documents_base_df)
print(f"Sorting Metric: Successfully ordered dataset records in {sort_duration:.4f} seconds.")
sorted_df.select("document_title", "path").show(5, truncate=False)

Sorting Metric: Successfully ordered dataset records in 2.8734 seconds.
+--------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------+
|document_title                                                                                                |path                                                                                                                                  |
+--------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------+
|Cloud Platform Architecture over Virtualized Data Center                                                      |dbfs:/Volumes/test/default/raw_docs_volume/highlighted_output/highlighte

In [0]:
import os

def search_and_highlight_documents(df, search_term, dbfs_output_dir):
    """
    Filters datasets matching text criteria phrases and compiles highlighted variants 
    of source assets onto cloud destinations.
    """
    start_time = time.time()
    
    # Distributed Text Search Filtering matching phrases
    matched_records_df = df.filter(col("document_text").like(f"%{search_term}%"))
    collected_matches = matched_records_df.select("path", "document_title").collect()
    
    os.makedirs(dbfs_output_dir, exist_ok=True)
    highlight_count = 0
    
    for row in collected_matches:
        source_path = row["path"]
        # Unity Catalog volumes use direct paths
        local_source_path = source_path.replace("dbfs:", "")
        file_name = os.path.basename(local_source_path)
        
        if local_source_path.lower().endswith('.pdf'):
            try:
                doc = fitz.open(local_source_path)
                term_found = False
                
                for page in doc:
                    text_instances = page.search_for(search_term)
                    for inst in text_instances:
                        term_found = True
                        page.add_highlight_annot(inst)
                
                if term_found:
                    destination_file_path = os.path.join(dbfs_output_dir, f"highlighted_{file_name}")
                    doc.save(destination_file_path, garbage=4, deflate=True)
                    highlight_count += 1
                doc.close()
            except Exception as e:
                print(f"[Error Processing File {file_name}]: {str(e)}")
                
        elif local_source_path.lower().endswith('.docx'):
            # Docx highlighting logic can be hooked here via native XML manipulation if required
            pass

    end_time = time.time()
    elapsed_time = end_time - start_time
    return matched_records_df, highlight_count, elapsed_time

# Run Search Engine Pipeline Execution
keyword = "Cloud Computing"
results_df, highlighted_files_written, search_duration = search_and_highlight_documents(
    documents_base_df, keyword, OUTPUT_HIGHLIGHTED_DIR
)

print(f"Search Metric: Discovered matches and generated ({highlighted_files_written}) highlighted targets within {search_duration:.4f} seconds.")

Search Metric: Discovered matches and generated (2) highlighted targets within 10.0377 seconds.


In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import DecisionTreeClassifier

# 1. Structural Mock Training Data Framework Generation (Predefined Target Classes Hierarchy)
# In production, replace this with your labeled training source files or Delta Table
training_dataset_schema = StructType([
    StructField("text_content", StringType(), True),
    StructField("category_label", StringType(), True)
])

mock_training_data = [
    ("A distributed cloud infrastructure deployment utilizing Apache Spark and Databricks architecture frameworks", "Technology"),
    ("Advanced software development patterns utilizing enterprise Java Spring Boot APIs and microservices systems", "Technology"),
    ("Clinical trials analyzing cardiovascular patient medical safety diagnostics metrics", "Healthcare"),
    ("Oncology protocols and pharmacological research for patient treatment management systems", "Healthcare"),
    ("Portfolio asset allocation analysis management metrics and risk mitigation forecasting models", "Finance"),
    ("Quarterly revenue growth statement analysis and stock corporate balance sheets evaluation models", "Finance")
]

training_df = spark.createDataFrame(mock_training_data, training_dataset_schema)

# 2. Design the End-to-End Spark ML Pipeline stages
tokenizer = Tokenizer(inputCol="text_content", outputCol="words")
sw_remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hashing_tf = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=5000)
idf = IDF(inputCol="raw_features", outputCol="features")
label_indexer = StringIndexer(inputCol="category_label", outputCol="indexed_label")

# The Machine Learning Tree implementation required by your specifications
decision_tree = DecisionTreeClassifier(labelCol="indexed_label", featuresCol="features")

classification_pipeline = Pipeline(stages=[tokenizer, sw_remover, hashing_tf, idf, label_indexer, decision_tree])

# 3. Model Training
print("[Status] Fitting Document Machine Learning Tree Pipeline...")
trained_model_pipeline = classification_pipeline.fit(training_df)

# 4. Ingest and Classify Unlabeled Live Production Documents
def classify_production_documents(model, target_df):
    start_time = time.time()
    
    # Format target source content to map internal training layer signatures
    inference_ready_df = target_df.withColumnRenamed("document_text", "text_content")
    
    # Run predictions
    predictions_df = model.transform(inference_ready_df)
    
    # Force action execution
    predictions_df.count()
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    return predictions_df, elapsed_time

# Run Classification (exclude highlighted_output files to prevent re-reading issues)
filtered_docs_df = documents_base_df.filter(~col("path").contains("highlighted_output"))
classified_output_df, classification_duration = classify_production_documents(trained_model_pipeline, filtered_docs_df)
print(f"Classification Metric: Document dataset categorized across clusters within {classification_duration:.4f} seconds.")

[Status] Fitting Document Machine Learning Tree Pipeline...
Classification Metric: Document dataset categorized across clusters within 13.2003 seconds.


In [0]:
from pyspark.sql.functions import sum as spark_sum

def display_analytics_dashboard_metrics():
    # Structural Storage Capacity calculations (exclude highlighted_output files)
    filtered_docs_df = documents_base_df.filter(~col("path").contains("highlighted_output"))
    total_space_bytes = filtered_docs_df.select(spark_sum("file_size_bytes")).collect()[0][0]
    total_space_mb = (total_space_bytes / (1024 * 1024)) if total_space_bytes else 0.0
    total_file_count = filtered_docs_df.count()
    
    
    print("DATA ANALYTICS SYSTEM METRICS OPERATIONAL REPORT")
    print(f" Storage Profile Total Documents Managed : {total_file_count} units")
    print(f" Total Aggregated Dataset Size          : {total_space_mb:.2f} MB ({total_space_bytes} Bytes)")
    print("")
    print("PIPELINE COMPUTE RUNTIME PROFILES")
    print(f" Document Title Sorting Sub-System      : {sort_duration:.4f} seconds")
    print(f" Text Pattern Search & PDF Highlight    : {search_duration:.4f} seconds")
    print(f" ML Decision Tree Batch Classification  : {classification_duration:.4f} seconds")

# Execute Dashboard Output Generation
display_analytics_dashboard_metrics()

DATA ANALYTICS SYSTEM METRICS OPERATIONAL REPORT
 Storage Profile Total Documents Managed : 19 units
 Total Aggregated Dataset Size          : 35.83 MB (37573546 Bytes)

PIPELINE COMPUTE RUNTIME PROFILES
 Document Title Sorting Sub-System      : 2.8734 seconds
 Text Pattern Search & PDF Highlight    : 10.0377 seconds
 ML Decision Tree Batch Classification  : 13.2003 seconds
